perch2_pytorch_verify.ipynb

Colab verification notebook for duane-edgington/perch2-pytorch-port

Runs on the free-tier CPU runtime — no GPU, no TensorFlow, no Google Drive needed.

Goal: take a new user from a fresh Colab session to a verified 1536-d
embedding with cosine parity reported against the ONNX reference.

Steps:
  1. Clone the repo
  2. Install dependencies (torch CPU, onnxruntime, huggingface_hub)
  3. Download the ONNX model (justinchuby/Perch-onnx, ~409 MB)
  4. Extract weights  -> perch_weights/weights.npz + graph_manifest.json
  5. Synthesize a 5-second test clip (32 kHz sine sweep — no audio file needed)
  6. Peak-normalize the clip (mandatory for quiet audio)
  7. Embed with the native PyTorch model
  8. Embed with the ONNX bridge (cross-check)
  9. Compare: cosine + relative L2 between native port and ONNX reference
 10. Summary / what to expect

Expected result on CPU:
  cosine(native, ONNX) > 0.9999  relative L2 < 1e-3
  (For a synthetic sine sweep, values may differ slightly from real audio.)

Required: Huggingface_Token placed in CoLab Secrets

https://huggingface.co/docs/hub/en/security-tokens



In [1]:
# =============================================================================
# Cell 1 — Clone the repo
# =============================================================================
# Expected output: "Already up to date." or a list of cloned files.

import subprocess, sys

result = subprocess.run(
    ["git", "clone", "--depth", "1",
     "https://github.com/duane-edgington/perch2-pytorch-port.git",
     "perch2-pytorch-port"],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)

import os, sys
repo = os.path.abspath("perch2-pytorch-port")
if repo not in sys.path:
    sys.path.insert(0, repo)
os.chdir(repo)
print("Working directory:", os.getcwd())
print("Repo files:", [f for f in os.listdir(".") if f.endswith(".py") or f.endswith(".md")])

Cloning into 'perch2-pytorch-port'...

Working directory: /content/perch2-pytorch-port
Repo files: ['perch_frontend_torch.py', 'perch_bench_native_spark.py', 'perch_hoplite_torch_adapter.py', 'extract_weights.py', 'perch_embedder_torch.py', 'README.md', 'benchmark_results.md', 'check_adapter.py', 'check_db.py', 'perch2_logmel_settings.md']


In [2]:
# =============================================================================
# Cell 2 — Install dependencies (CPU-only; ~2 min on first run)
# =============================================================================
# torch, onnxruntime, huggingface_hub.  No TensorFlow needed.

import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "torch", "onnxruntime", "huggingface_hub", "numpy"], check=True)
print("Dependencies ready.")


# =============================================================================
# Cell 3 — Download the ONNX model from HuggingFace (~409 MB, ~30-60 s)
# =============================================================================
# justinchuby/Perch-onnx: a faithful ONNX export of Google's Perch 2.0 TF model.
# Used here as the weight source and as the numerical reference.
# Apache-2.0.  We do NOT redistribute the model — it is fetched at runtime.

from huggingface_hub import hf_hub_download

print("Downloading perch_v2.onnx from justinchuby/Perch-onnx ...")
onnx_path = hf_hub_download(
    repo_id   = "justinchuby/Perch-onnx",
    filename  = "perch_v2.onnx",
    repo_type = "model",
)
print(f"Downloaded to: {onnx_path}")
print(f"File size:     {os.path.getsize(onnx_path)/1e6:.1f} MB")

Dependencies ready.


perch_v2.onnx: reconstructing file:   0%|          |  0.00B /  409MB            

perch_v2.onnx: downloading bytes:           |  0.00B            

Downloaded to: /root/.cache/huggingface/hub/models--justinchuby--Perch-onnx/snapshots/da88bf8c37c0b3068fc706fe509f633b9fbd28c1/perch_v2.onnx
File size:     409.1 MB


In [3]:
# =============================================================================
# Cell 3 — Download the ONNX model from HuggingFace (~409 MB, ~30-60 s)
# =============================================================================
# justinchuby/Perch-onnx: a faithful ONNX export of Google's Perch 2.0 TF model.
# Used here as the weight source and as the numerical reference.
# Apache-2.0.  We do NOT redistribute the model — it is fetched at runtime.

from huggingface_hub import hf_hub_download

print("Downloading perch_v2.onnx from justinchuby/Perch-onnx ...")
onnx_path = hf_hub_download(
    repo_id   = "justinchuby/Perch-onnx",
    filename  = "perch_v2.onnx",
    repo_type = "model",
)
print(f"Downloaded to: {onnx_path}")
print(f"File size:     {os.path.getsize(onnx_path)/1e6:.1f} MB")

Downloaded to: /root/.cache/huggingface/hub/models--justinchuby--Perch-onnx/snapshots/da88bf8c37c0b3068fc706fe509f633b9fbd28c1/perch_v2.onnx
File size:     409.1 MB


In [5]:
# =============================================================================
# Cell 4 — Extract weights from the ONNX model
# =============================================================================
# Runs perch_extract_weights_spark.py (despite the name, works on CPU/Colab).
# Writes:  ./perch_weights/weights.npz  (~45 MB)
#          ./perch_weights/graph_manifest.json
# This is the only step that reads the ONNX model; after this the native
# PyTorch model is entirely TF/ONNX-free.

weights_dir = os.path.join(repo, "perch_weights")
os.makedirs(weights_dir, exist_ok=True)

print("Extracting weights (may take 30-60 s on CPU) ...")
result = subprocess.run(
    [sys.executable, "extract_weights.py",
     "--onnx",    onnx_path,
     "--out_dir", weights_dir],
    capture_output=True, text=True
)
print(result.stdout[-2000:] if result.stdout else "")
if result.returncode != 0:
    print("STDERR:", result.stderr[-1000:])
    raise RuntimeError("Weight extraction failed — see stderr above.")

weights_npz  = os.path.join(weights_dir, "weights.npz")
manifest     = os.path.join(weights_dir, "graph_manifest.json")
print(f"\nweights.npz :       {os.path.getsize(weights_npz)/1e6:.1f} MB")
print(f"graph_manifest.json: {os.path.getsize(manifest)/1e3:.1f} KB")
print("Weight extraction complete.")

Extracting weights (may take 30-60 s on CPU) ...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 19.4 MB/s eta 0:00:00
graph: 1137 nodes, 442 initializers
saved 441 weight tensors (11.3M params) -> weights.npz (45.3 MB)
skipped 1 big tensors (class heads): ['jit(infer_fn)/MultiHeadClassifier/MultiHeadClassifier._call_model/heads_protopnet_logits/dot_general6_reshaped_0']

Weights written to /content/perch2-pytorch-port/perch_weights/ — the adapter reads this directory directly.


weights.npz :       45.3 MB
graph_manifest.json: 347.0 KB
Weight extraction complete.


In [6]:
# =============================================================================
# Cell 5 — Synthesize a 5-second test clip (no audio file required)
# =============================================================================
# A 32 kHz mono sine sweep from 500 Hz to 4 kHz.
# Peak is ~0.002 (quiet, like a deep-water hydrophone recording) to exercise
# the low-amplitude path that motivated the per-window normalization finding.

import numpy as np

SR    = 32_000
DUR   = 5.0
N     = int(SR * DUR)        # 160,000 samples
t     = np.linspace(0, DUR, N, endpoint=False)
f0, f1 = 500.0, 4000.0      # sweep from 500 Hz to 4 kHz
phase = 2 * np.pi * (f0 * t + 0.5 * (f1 - f0) / DUR * t**2)
clip  = (0.002 * np.sin(phase)).astype(np.float32)  # quiet: peak ~0.002

print(f"Test clip: {N} samples @ {SR} Hz  ({DUR} s)")
print(f"Peak amplitude: {np.abs(clip).max():.5f}  (quiet — mimics deep-water hydrophone)")


Test clip: 160000 samples @ 32000 Hz  (5.0 s)
Peak amplitude: 0.00200  (quiet — mimics deep-water hydrophone)


In [7]:
# =============================================================================
# Cell 6 — Peak-normalize to 0.25 (mandatory for low-amplitude audio)
# =============================================================================
# Perch 2.0's log-mel frontend computes 0.1*log(max(mel, 1e-5)).  For quiet
# audio (peak << 0.05) the mel bins saturate the log floor, causing the
# frontend's arithmetic to diverge from the reference.  Peak-normalizing to
# 0.25 before embedding is the fix — and it is applied by default inside
# PerchModel when you call peak_normalize=True (the default).
# We show it explicitly here so the notebook is pedagogically clear.

def peak_normalize(x, target=0.25):
    """Scale audio so peak absolute value == target.  Float64 intermediate."""
    x64  = x.astype(np.float64)
    peak = np.abs(x64).max()
    return (x64 * (target / peak)).astype(np.float32) if peak > 1e-8 else x.astype(np.float32)

clip_norm = peak_normalize(clip)
print(f"Before normalization: peak = {np.abs(clip).max():.5f}")
print(f"After  normalization: peak = {np.abs(clip_norm).max():.5f}  (should be 0.25)")

Before normalization: peak = 0.00200
After  normalization: peak = 0.25000  (should be 0.25)


In [8]:
# =============================================================================
# Cell 7 — Embed with the native PyTorch port
# =============================================================================
# PerchModel(weights_dir) loads weights.npz + graph_manifest.json and builds
# the full raw-audio -> embedding pipeline as an nn.Module.
# No TensorFlow, no ONNX runtime — pure PyTorch.

import torch
from perch_embedder_torch import PerchModel

print("Loading native PyTorch model ...")
model = PerchModel(weights_dir).eval()
print(f"Model device: {next(model.parameters()).device}")

audio_tensor = torch.from_numpy(clip_norm)   # (160000,)

with torch.no_grad():
    emb_native = model(audio_tensor).numpy().reshape(-1)

print(f"\nNative PyTorch embedding:")
print(f"  shape:  {emb_native.shape}")
print(f"  norm:   {np.linalg.norm(emb_native):.4f}")
print(f"  first 5 values: {emb_native[:5]}")

Loading native PyTorch model ...
Model device: cpu

Native PyTorch embedding:
  shape:  (1536,)
  norm:   4.3358
  first 5 values: [-0.02457603 -0.05932628  0.02764375  0.05296747 -0.10749799]


In [9]:
# =============================================================================
# Cell 8 — Embed with the ONNX bridge (cross-check reference)
# =============================================================================
# onnxruntime runs the original ONNX graph — same weights, different runtime.
# This is the numerical reference we compare against.
# Note: the ONNX bridge takes raw audio and does NOT apply peak-norm internally,
# so we feed it the same already-normalized clip.

import onnxruntime as ort

print("Loading ONNX model into onnxruntime (CPU) ...")
sess    = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
in_name = sess.get_inputs()[0].name
print(f"ONNX input name: '{in_name}' shape: {sess.get_inputs()[0].shape}")

emb_onnx = sess.run(
    ["embedding"],
    {in_name: clip_norm.reshape(1, -1)}
)[0].reshape(-1)

print(f"\nONNX bridge embedding:")
print(f"  shape:  {emb_onnx.shape}")
print(f"  norm:   {np.linalg.norm(emb_onnx):.4f}")
print(f"  first 5 values: {emb_onnx[:5]}")


Loading ONNX model into onnxruntime (CPU) ...
ONNX input name: 'inputs' shape: ['batch', 160000]

ONNX bridge embedding:
  shape:  (1536,)
  norm:   4.3358
  first 5 values: [-0.02457621 -0.05932601  0.02764371  0.05296743 -0.10749788]


In [10]:
# =============================================================================
# Cell 9 — Compare native port vs ONNX reference
# =============================================================================

def cosine(a, b):
    a, b = a.reshape(-1), b.reshape(-1)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def rel_l2(a, b):
    a, b = a.reshape(-1), b.reshape(-1)
    return float(np.linalg.norm(a - b) / np.linalg.norm(b))

cos_val  = cosine(emb_native, emb_onnx)
rl2_val  = rel_l2(emb_native, emb_onnx)
pass_cos = cos_val  > 0.9990
pass_rl2 = rl2_val  < 0.010

print("=" * 60)
print("  Native PyTorch port  vs  ONNX bridge (cross-check)")
print("=" * 60)
print(f"  cosine similarity : {cos_val:.7f}   {'PASS ✓' if pass_cos else 'FAIL ✗'}")
print(f"  relative L2 error : {rl2_val:.3e}   {'PASS ✓' if pass_rl2 else 'FAIL ✗'}")
print("=" * 60)

if pass_cos and pass_rl2:
    print("\n  ✓  Port verified: embeddings agree with the ONNX reference.")
    print("     (Cosine > 0.999 and relative L2 < 1e-2 on a synthetic clip.)")
    print("     On real audio the cosine is typically > 0.9999.")
else:
    print("\n  ✗  Something looks off — check cells above for errors.")


  Native PyTorch port  vs  ONNX bridge (cross-check)
  cosine similarity : 1.0000001   PASS ✓
  relative L2 error : 3.174e-06   PASS ✓

  ✓  Port verified: embeddings agree with the ONNX reference.
     (Cosine > 0.999 and relative L2 < 1e-2 on a synthetic clip.)
     On real audio the cosine is typically > 0.9999.


What this notebook verified
---------------------------
1. The repo clones cleanly from GitHub.
2. Weights extract correctly from the ONNX model (justinchuby/Perch-onnx).
3. PerchModel loads those weights and produces a 1536-d embedding from a
   synthetic 5-second clip — no TensorFlow, no GPU required.
4. The embedding agrees with the ONNX bridge at cosine > 0.999.

This is a SYNTHETIC test clip (sine sweep, peak 0.002).  The cosine against
the ONNX reference will typically be even closer (> 0.9999) on real audio.

Validated results on real audio (MARS hydrophone clips, GB10 GPU):
  cosine vs reference TF  :  0.9999998   (4 clips, Sept 2026)
  relative L2 vs ref TF   :  ~4e-4       (GPU; ~1-5e-5 on CPU)

See README.md and dev/ for the full validation story.

Repo  : https://github.com/duane-edgington/perch2-pytorch-port
ONNX  : https://huggingface.co/justinchuby/Perch-onnx
Science: https://github.com/duane-edgington/perch-hoplite-orcas-MBNMS
